# Regresi kuantil (interval prediksi)
**tabular › regression_advanced** · notebook `quantile_interval`

**Kapan dipakai:** Butuh rentang (mis. 90%) di samping tebakan titik, atau metrik pinball/interval dinilai.

**Jebakan umum:** Kuantil yang dilatih terpisah bisa saling menyilang. Periksa coverage nyata di validation, jangan percaya nominalnya.

## 0. Setup
Import dan lokasi proyek. Tidak perlu diubah.

In [ ]:
import sys, pathlib, warnings
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src" / "superdb").is_dir())
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", 50)
from superdb.common import *
from superdb.paths import out_dir

## 1. Konfigurasi
Ganti bagian ini saja saat memakai data lomba.

In [ ]:
TRAIN_PATH = ROOT / "data_latihan/tabular/regression/diamonds_train.csv"
TEST_PATH  = ROOT / "data_latihan/tabular/regression/diamonds_test.csv"      # None kalau tidak ada file test
TARGET     = "price"
ID_COL     = 'id'
DROP_COLS  = []
GROUP_COL  = None                      # isi bila ada grup agar tidak bocor
SCORING    = "neg_root_mean_squared_error"
SEED, N_SPLITS, VALID_SIZE = 42, 5, 0.2
MODEL_KEY  = "quantile_interval"
TASK       = "regression"
KIND       = "tree"
OUT = out_dir("tabular", "regression_advanced", MODEL_KEY)
set_seed(SEED)

## 2. Baca data

In [ ]:
df = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH) if TEST_PATH else None
print(df.shape, None if test is None else test.shape)
display(df.head(3))

## 3. Fitur & split

In [ ]:
from superdb.tabular import detect_columns, prepare_features, build_pipeline, split_train_valid, cv_score
cols = detect_columns(df, target=TARGET, id_col=ID_COL, drop_cols=DROP_COLS)
X, NUM, CAT = prepare_features(df, cols)
y = df[TARGET].to_numpy(dtype=float)
groups = df[GROUP_COL] if GROUP_COL else None
Xtr, Xva, ytr, yva = split_train_valid(X, y, TASK, VALID_SIZE, SEED, groups)
GTR = None if groups is None else groups.iloc[Xtr.index]   # grup untuk CV
print(Xtr.shape, Xva.shape)

## 4. Model dasar

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.base import clone
lgbm = LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=31, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEED, n_jobs=-1, verbose=-1)
def fit_eval(name, model_, kind="tree", ytr_=None, store=None):
    pp = build_pipeline(model_, NUM, CAT, kind=kind).fit(Xtr, ytr if ytr_ is None else ytr_)
    pr = pp.predict(Xva)
    if store is not None: store[name] = pr
    return {"model": name, **regression_metrics(yva, pr)}

## 5. Tiga model kuantil: 5%, 50%, 95% (interval nominal 90%)

In [ ]:
from superdb.tabular_extra_b import interval_coverage
QS = {"lo": 0.05, "mid": 0.5, "hi": 0.95}
qm = {k: build_pipeline(LGBMRegressor(objective="quantile", alpha=a, n_estimators=300, learning_rate=0.05, num_leaves=31, random_state=SEED, n_jobs=-1, verbose=-1), NUM, CAT, kind="tree").fit(Xtr, ytr) for k, a in QS.items()}
q = {k: mm.predict(Xva) for k, mm in qm.items()}
lo, hi = np.minimum(q["lo"], q["hi"]), np.maximum(q["lo"], q["hi"])          # cegah kuantil menyilang
cov, width = interval_coverage(yva, lo, hi)
print(f"coverage nyata = {cov:.3f} (nominal 0.90) | lebar rata-rata = {width:,.1f} | persentase kuantil menyilang = {(q['lo'] > q['hi']).mean():.3%}")
metrics = {**regression_metrics(yva, q["mid"]), "coverage_90": cov, "mean_width": width}
display(show_table(metrics))

In [ ]:
o = np.argsort(q["mid"])[:: max(1, len(yva) // 120)]
fig, ax = plt.subplots(figsize=(9, 4))
ax.fill_between(range(len(o)), lo[o], hi[o], alpha=.3, label="interval 90%"); ax.plot(q["mid"][o], "b-", lw=1, label="median"); ax.plot(np.asarray(yva)[o], "k.", ms=4, label="aktual")
ax.legend(); ax.set_title("Interval prediksi (diurut menurut prediksi median)"); plt.tight_layout(); plt.savefig(OUT / "quantile_interval.png", dpi=130); plt.show()

In [ ]:
try:
    save_json({"method": MODEL_KEY, "valid": res.reset_index().to_dict("records") if "res" in dir() else metrics}, OUT / "metrics.json")
except NameError:
    save_json({"method": MODEL_KEY, "valid": metrics}, OUT / "metrics.json")
print("OK. hasil di", OUT)

## Langkah berikutnya
- Bandingkan metrik pada skala yang dipakai juri lomba, bukan hanya RMSE.
- Hasil terbaik biasanya digabung ke ensemble (lihat `notebooks/tabular/ensembling_regression/`).